# BananaClock - Model Exploration & Evaluation

This notebook is for:
- Exploring trained models
- Evaluating model performance
- Testing predictions on sample images
- Visualizing model outputs

## Setup

In [ ]:
# Install dependencies
!pip install -q ultralytics tensorflow pillow matplotlib seaborn

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from ultralytics import YOLO
import os

print("TensorFlow:", tf.__version__)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Model paths
MODELS_DIR = "/content/drive/MyDrive/BananaClock/models"
CLASSIFIER_PATH = f"{MODELS_DIR}/banana_classifier.h5"
YOLO_PATH = f"{MODELS_DIR}/banana_yolo.pt"

# Check if models exist (with fallback for YOLO)
if not os.path.exists(YOLO_PATH):
    print("Standard YOLO path not found, checking training output folder...")
    YOLO_PATH = f"{MODELS_DIR}/banana_yolo/weights/best.pt"

print("Classifier exists:", os.path.exists(CLASSIFIER_PATH))
print("YOLO exists:", os.path.exists(YOLO_PATH))
print(f"Using YOLO Path: {YOLO_PATH}")

## Load Models

In [ ]:
# Load classification model
classifier = tf.keras.models.load_model(CLASSIFIER_PATH)
print("Classifier loaded!")
classifier.summary()

In [ ]:
# Load YOLO model
yolo = YOLO(YOLO_PATH)
print("YOLO loaded!")

## Classes and Helper Functions

In [ ]:
# Define classes
CLASSES = ["fresh", "slightly_ripe", "ripe", "overripe", "spoiled"]
CLASS_COLORS = {
    "fresh": "#059669",
    "slightly_ripe": "#84cc16",
    "ripe": "#f59e0b",
    "overripe": "#ea580c",
    "spoiled": "#ef4444"
}
DAYS_MAPPING = {
    "fresh": 7,
    "slightly_ripe": 5,
    "ripe": 2,
    "overripe": 1,
    "spoiled": 0
}

def preprocess_image(image_path, target_size=(224, 224)):
    """Preprocess image for classification."""
    img = Image.open(image_path).convert("RGB")
    img = img.resize(target_size, Image.Resampling.LANCZOS)
    img_array = np.array(img) / 255.0
    return np.expand_dims(img_array, axis=0)

def predict_banana(image_path):
    """Run full prediction pipeline."""
    # YOLO detection
    yolo_results = yolo(image_path)[0]
    boxes = yolo_results.boxes
    
    # If no detection, use full image
    if len(boxes) == 0:
        print("No bananas detected, using full image")
        input_img = preprocess_image(image_path)
        predictions = classifier.predict(input_img, verbose=0)
        class_idx = np.argmax(predictions[0])
        confidence = predictions[0][class_idx]
        return [{
            "condition": CLASSES[class_idx],
            "confidence": float(confidence),
            "days_until_bad": DAYS_MAPPING[CLASSES[class_idx]]
        }]
    
    # Process each detected banana
    results = []
    img = Image.open(image_path).convert("RGB")
    
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        cropped = img.crop((x1, y1, x2, y2))
        cropped = cropped.resize((224, 224), Image.Resampling.LANCZOS)
        input_img = np.expand_dims(np.array(cropped) / 255.0, axis=0)
        
        predictions = classifier.predict(input_img, verbose=0)
        class_idx = np.argmax(predictions[0])
        confidence = predictions[0][class_idx]
        
        results.append({
            "banana_id": i + 1,
            "condition": CLASSES[class_idx],
            "confidence": float(confidence),
            "days_until_bad": DAYS_MAPPING[CLASSES[class_idx]],
            "bbox": [x1, y1, x2, y2]
        })
    
    return results

## Test on Sample Images

In [ ]:
# Upload test image
from google.colab import files
uploaded = files.upload()

test_image = list(uploaded.keys())[0]
print(f"Testing with: {test_image}")

In [ ]:
# Run prediction
results = predict_banana(test_image)

# Display results
print("\n=== Prediction Results ===")
for r in results:
    print(f"\nBanana {r.get('banana_id', 1)}:")
    print(f"  Condition: {r['condition'].upper()}")
    print(f"  Confidence: {r['confidence']*100:.1f}%")
    print(f"  Days until bad: {r['days_until_bad']}")

In [ ]:
# Visualize prediction
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original image
img = Image.open(test_image)
axes[0].imshow(img)
axes[0].set_title("Original Image")
axes[0].axis("off")

# YOLO detection
yolo_results = yolo(test_image)[0]
annotated = yolo_results.plot()
axes[1].imshow(annotated)
axes[1].set_title(f"Detection: {len(yolo_results.boxes)} banana(s)")
axes[1].axis("off")

plt.tight_layout()
plt.show()

# Class probabilities
fig, ax = plt.subplots(figsize=(8, 4))
input_img = preprocess_image(test_image)
probs = classifier.predict(input_img, verbose=0)[0]

colors = [CLASS_COLORS[c] for c in CLASSES]
bars = ax.barh(CLASSES, probs * 100, color=colors)
ax.set_xlabel("Confidence (%)")
ax.set_title("Classification Probabilities")
ax.set_xlim(0, 100)

for bar, prob in zip(bars, probs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f"{prob*100:.1f}%", va='center')
plt.tight_layout()
plt.show()

## Test on Dataset

In [ ]:
# Test on multiple images from test set
TEST_DATA_DIR = "/content/drive/MyDrive/BananaClock/processed_data/test"

if os.path.exists(TEST_DATA_DIR):
    correct = 0
    total = 0
    
    for class_name in CLASSES:
        class_dir = f"{TEST_DATA_DIR}/{class_name}"
        if not os.path.exists(class_dir):
            continue
            
        images = os.listdir(class_dir)[:10]  # Sample 10 per class
        for img_name in images:
            img_path = f"{class_dir}/{img_name}"
            try:
                results = predict_banana(img_path)
                predicted = results[0]["condition"]
                if predicted == class_name:
                    correct += 1
                total += 1
            except Exception as e:
                print(f"Error: {e}")
    
    print(f"\nSample Test Accuracy: {correct}/{total} = {correct/total*100:.1f}%")
else:
    print("Test data not found. Run training first.")

## Model Analysis

In [ ]:
# Classifier layer analysis
print("=== Classifier Architecture ===")
for i, layer in enumerate(classifier.layers):
    print(f"{i:2d}. {layer.name:30s} - {layer.output_shape}")

In [ ]:
# YOLO model info
print("=== YOLO Model Info ===")
print(yolo.info())